# NDR — Notebook 4-v2: Retain-THẬT (ảnh test) + Prune cls+box + FT-SAM

**Deliverable:** model sửa weight, gỡ backdoor, GIỮ detect streak. Submit output TRỰC TIẾP. Không boost, không rescoring.

## Điểm neo LB thật (Version 28)
prune-only 5% → **LB maCADD = 261.2** (empty=284, poisoned=379, top≈145). Prune-only là thật, không collapse (0.37 dets/img). Local val_macadd 1.093 đi CÙNG chiều LB → local là proxy tin cậy để tune nhanh.

## Vì sao NB3 chưa đủ (bằng chứng sweep)
firing & retention tụt CÙNG nhau: prune5%→ret0.55, prune10%→ret0.20. Trên cls_subnet RetinaNet, neuron bắn poison bám CHUNG đường detect streak → prune mù không phải dao mổ đủ sắc (khớp arXiv 2409.16057: defense kiểu classifier hay fail trên detector).

## 3 cải tiến từ research
1. **Retain set DÀY, in-distribution** — pseudo-GT streak thật từ N ảnh TEST (conf cao, bỏ box trùng poison), thay vì 20 ảnh poison (0.25 box/ảnh gây collapse). arXiv 2502.07231: auxiliary data in-distribution là điều kiện sống còn để purify.
2. **FT-SAM (sharpness-aware) thay SGD** — arXiv 2304.11823: fine-tune thường thất bại vì neuron backdoor kẹt local minimum & có norm lớn; SAM co norm chúng → gỡ poison mà giữ clean, đúng chế độ ít dữ liệu.
3. **EWC anchor** giữ head không trôi khỏi model gốc (chống over-unlearn).

**Quy tắc nộp:** chỉ nộp khi local val_macadd < 1.093 (điểm neo hiện tại) VÀ retention ≥ 0.6.

**v2 cải tiến (theo A/B NB8 + research):**
- Retain set từ ẢNH TEST THẬT (pseudo-GT model gốc) + synth bơm mật độ — thay synthetic thuần (retention kẹt 0.59).
- Prune SOI CẢ cls_subnet VÀ bbox_subnet (arXiv 2409.16057), không chỉ cls.
- Mục tiêu: kéo retention 0.59 -> ~0.75+ mà giữ firing thấp -> phá LB 254.

## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 804.9 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 2.5 MB/s eta 0:00:00


In [2]:
import copy, json, itertools, pickle, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.structures import BoxMode
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB  = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
SUBMISSION_PATH = str(WORK / 'submission.csv')

BASE_CONFIG          = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

A_FACTOR = 10.0
IOU_THRESHOLDS = np.round(np.arange(0.2, 0.91, 0.1), 2)
CAND_THRESH = 0.05
SUBMIT_THRESH = 0.2
CLEAN_CONF_FLOOR = 0.2
MATCH_IOU = 0.3
POISON_OVERLAP_IOU = 0.1   # detection trùng poison nếu IoU>=0.1 (loại khỏi retain)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE)

Setup OK | device: cuda


## 2. Metric core — maCADD (unit-tested)

In [3]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)), np.float32)
    ix1 = np.maximum(a[:,0:1], b[None,:,0]); iy1 = np.maximum(a[:,1:2], b[None,:,1])
    ix2 = np.minimum(a[:,2:3], b[None,:,2]); iy2 = np.minimum(a[:,3:4], b[None,:,3])
    iw = np.clip(ix2-ix1,0,None); ih = np.clip(iy2-iy1,0,None); inter = iw*ih
    aa = ((a[:,2]-a[:,0])*(a[:,3]-a[:,1]))[:,None]; ab = ((b[:,2]-b[:,0])*(b[:,3]-b[:,1]))[None,:]
    return np.where(aa+ab-inter>0, inter/(aa+ab-inter+1e-9), 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip):
    # SPEC BTC: mỗi clean box khớp de-pois box IoU CAO NHẤT (>=t); các de-pois box thừa = FP.
    ious = iou_matrix(cb, pb); cost = 0.0; mc=set(); mp=set()
    if ious.size:
        pairs = sorted(((ious[i,j], i, j) for i in range(len(cb)) for j in range(len(pb)) if ious[i,j] >= t), reverse=True)
        for _, i, j in pairs:
            if i in mc or j in mp: continue
            mc.add(i); mp.add(j); d = float(cs[i]) - float(ps[j])
            if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
            else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)   # FN: clean chưa khớp
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)   # FP: de-pois box thừa
    return cost

def macadd(clean_ref, pred, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for key,(cb,cs) in clean_ref.items():
        keep = cs>CLEAN_CONF_FLOOR; cb2,cs2 = cb[keep], cs[keep]
        cip = (poison_flags.get(key, np.zeros(len(cb),bool))[keep] if poison_flags else np.zeros(len(cs2),bool))
        pb,ps = pred.get(key,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        per.append(sum(wi*acadd_t(cb2,cs2,pb,ps,t,cip) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per)) if per else 0.0

bx = lambda x,y,w,h: np.array([[x,y,x+w,y+h]],np.float32); E=(np.zeros((0,4),np.float32),np.zeros(0,np.float32))
cl = {'i':(bx(10,10,40,40), np.array([0.6],np.float32))}
assert abs(macadd(cl,{'i':E})-0.6)<1e-6                                   # miss streak = FN full
assert abs(macadd(cl,{'i':(bx(10,10,40,40),np.array([0.2]))})-0.4)<1e-6   # undershoot conf = full
assert abs(macadd({'i':E},{'i':(bx(5,5,20,20),np.array([0.5]))})-0.5)<1e-6 # fire on nothing = FP full
print('maCADD unit tests passed')

maCADD unit tests passed


## 3. Load data + poisoned model

In [4]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype==np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im
def build_cfg(weights, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]; cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh; cfg.MODEL.DEVICE = DEVICE; cfg.INPUT.FORMAT = 'RGB'
    if output_dir: cfg.OUTPUT_DIR = str(output_dir); Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))
def predict_image(predictor, img):
    inst = predictor(img)['instances'].to('cpu')
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco = json.load(f)
stem_by_id = {im['id']: Path(im['file_name']).stem for im in coco['images']}
poison_boxes_by_stem = {}
for a in coco['annotations']:
    x,y,w,h = a['bbox']; poison_boxes_by_stem.setdefault(stem_by_id[a['image_id']], []).append([x,y,x+w,y+h])
poison_boxes_by_stem = {k: np.array(v,np.float32) for k,v in poison_boxes_by_stem.items()}
poison_xywh = {}
for a in coco['annotations']: poison_xywh.setdefault(a['image_id'], []).append(a['bbox'])
unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'), key=lambda p:int(p.stem))
unlearn_imgs = {p.stem: load_image(p) for p in unlearn_files}
test_files = sorted(Path(TEST_DIR).glob('*.png'), key=lambda p:int(p.stem))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print('unlearn:', len(unlearn_files), '| test:', len(test_files))

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


unlearn: 20 | test: 2000


## 4. Validation set (poison + synth streak thật)

In [5]:
MARGIN = 6
poison_patches = []
for a in coco['annotations']:
    x,y,w,h = a['bbox']; stem = stem_by_id[a['image_id']]
    x0,y0 = int(max(0,x-MARGIN)), int(max(0,y-MARGIN)); x1,y1 = int(min(IMG_W,x+w+MARGIN)), int(min(IMG_H,y+h+MARGIN))
    poison_patches.append(unlearn_imgs[stem][y0:y1, x0:x1].copy())
stems = list(unlearn_imgs)
def rand_free_spot(stem, ph, pw, rng, tries=50):
    pb = poison_boxes_by_stem.get(stem, np.array([[0,0,1,1]],np.float32))
    for _ in range(tries):
        x0 = rng.integers(16, IMG_W-pw-16); y0 = rng.integers(16, IMG_H-ph-16)
        if iou_matrix(np.array([[x0,y0,x0+pw,y0+ph]],np.float32), pb).max()<0.01: return int(x0),int(y0)
    return None
def paste_patch(img, patch, x0, y0, feather=3):
    ph,pw = patch.shape[:2]; reg = img[y0:y0+ph, x0:x0+pw]
    if reg.shape[:2]!=(ph,pw): return
    mask = np.ones((ph,pw,1),np.float32)
    if ph>2*feather and pw>2*feather:
        rh = np.minimum(np.arange(ph)[:,None]+1, np.arange(ph)[::-1][:,None]+1)/feather
        rw = np.minimum(np.arange(pw)[None,:]+1, np.arange(pw)[::-1][None,:]+1)/feather
        mask = np.clip(np.minimum(rh,rw),0,1)[...,None]
    img[y0:y0+ph, x0:x0+pw] = reg*(1-mask) + patch*mask
def synth_streak_on(bg, rng):
    h,w = bg.shape; canvas = np.zeros_like(bg)
    ang = rng.uniform(0,np.pi); L = rng.uniform(20,110); cx,cy = rng.uniform(0.25,0.75)*w, rng.uniform(0.25,0.75)*h
    p1 = (int(cx-L/2*np.cos(ang)), int(cy-L/2*np.sin(ang))); p2 = (int(cx+L/2*np.cos(ang)), int(cy+L/2*np.sin(ang)))
    cv2.line(canvas,p1,p2,1.0,1); canvas = cv2.GaussianBlur(canvas,(0,0),rng.uniform(0.6,1.6))
    amp = rng.uniform(3,14)*(np.std(bg)+1e-3); out = bg + canvas/(canvas.max()+1e-6)*amp
    box = [min(p1[0],p2[0])-3, min(p1[1],p2[1])-3, max(p1[0],p2[0])+3, max(p1[1],p2[1])+3]
    return out, [float(np.clip(box[0],0,w)),float(np.clip(box[1],0,h)),float(np.clip(box[2],0,w)),float(np.clip(box[3],0,h))]
def build_validation_set(n=80, seed=SEED):
    rng = np.random.default_rng(seed); val=[]
    for _ in range(n):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); streaks=[]
        for _ in range(rng.integers(1,4)):
            g2,box = synth_streak_on(img[:,:,0], rng); img = np.repeat(g2[:,:,None],3,axis=2); streaks.append(box)
        pois = [list(map(float,b)) for b in poison_boxes_by_stem.get(stem, [])]
        for _ in range(rng.integers(0,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem,ph,pw,rng)
            if spot: x0,y0 = spot; paste_patch(img,patch,x0,y0); pois.append([x0+MARGIN,y0+MARGIN,x0+pw-MARGIN,y0+ph-MARGIN])
        val.append({'img':img,'streak_boxes':streaks,'poison_boxes':pois})
    return val
VAL = build_validation_set(80)
print('val images:', len(VAL), '| streaks:', sum(len(v['streak_boxes']) for v in VAL), '| poison:', sum(len(v['poison_boxes']) for v in VAL))

val images: 80 | streaks: 159 | poison: 163


## 5. Clean reference = CHỈ streak thật

In [6]:
def make_clean_reference(val, predictor):
    '''clean_ref = CHỈ streak thật (khớp detection model gốc để lấy conf tham chiếu).
    Poison KHÔNG vào ref -> mọi detection trên poison sẽ là FP.'''
    ref = {}
    for i,item in enumerate(val):
        b,s = predict_image(predictor, item['img'])
        rb,rs = [],[]
        sb = np.array(item['streak_boxes'],np.float32).reshape(-1,4)
        for k in range(len(sb)):
            if len(b):
                ious = iou_matrix(sb[k:k+1], b)[0]; j = ious.argmax()
                if ious[j]>=MATCH_IOU: rb.append(b[j]); rs.append(float(s[j]))
        ref[str(i)] = (np.array(rb,np.float32).reshape(-1,4), np.array(rs,np.float32))
    return ref
CLEAN_REF = make_clean_reference(VAL, poisoned_predictor)
print('clean reference (streak-only) built for', len(CLEAN_REF), 'val images')

clean reference (streak-only) built for 80 val images


## 6. Thước đo genuine + TTA helpers

In [7]:
def eval_model_on_val(predictor, val, tta=False):
    fired=ft=0; kept=st=0; pk={}
    for i,item in enumerate(val):
        b,s = (predict_tta(predictor, item['img']) if tta else predict_image(predictor, item['img']))
        keep = s>SUBMIT_THRESH; b2,s2 = (b[keep], s[keep]) if len(b) else (b,s); pk[str(i)]=(b2,s2)
        for pbx in item['poison_boxes']:
            ft+=1
            if len(b2) and iou_matrix(np.array([pbx],np.float32),b2).max()>=MATCH_IOU: fired+=1
        for sbx in item['streak_boxes']:
            st+=1
            if len(b2) and iou_matrix(np.array([sbx],np.float32),b2).max()>=MATCH_IOU: kept+=1
    return dict(poison_firing_rate=fired/max(ft,1), streak_retention=kept/max(st,1), val_macadd=macadd(CLEAN_REF, pk))

base_stats = eval_model_on_val(poisoned_predictor, VAL)
print('POISONED (chưa xử lý):', {k: round(v,4) for k,v in base_stats.items()})
print('-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.')
print('   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.')

POISONED (chưa xử lý): {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
-> baseline: firing cao (bắn poison=FP), retention=1, val_macadd = mức của model gốc.
   Mục tiêu: firing↓ (bớt FP poison) mà retention giữ -> val_macadd < baseline.


In [8]:
D4 = [(k,f) for f in (0,1) for k in range(4)]
def apply_d4(img,k,flip):
    x = np.rot90(img,k).copy() if k else img
    return x[:, ::-1, :].copy() if flip else x
def invert_d4(boxes,k,flip,H,W):
    S = W; b = boxes.astype(np.float32).copy()
    if flip: x1 = S-b[:,2]; x2 = S-b[:,0]; b[:,0],b[:,2] = x1,x2
    for _ in range(k%4):
        x1=b[:,0].copy(); y1=b[:,1].copy(); x2=b[:,2].copy(); y2=b[:,3].copy()
        b[:,0]=S-y2; b[:,1]=x1; b[:,2]=S-y1; b[:,3]=x2
    return b
def predict_tta(predictor, img):
    H,W = img.shape[:2]; bl,sl,ll = [],[],[]
    for k,flip in D4:
        b,s = predict_image(predictor, apply_d4(img,k,flip))
        if len(b)==0: continue
        b = invert_d4(b,k,flip,H,W); bn = b.copy()
        bn[:,[0,2]] = np.clip(bn[:,[0,2]],0,W)/W; bn[:,[1,3]] = np.clip(bn[:,[1,3]],0,H)/H
        bl.append(bn.tolist()); sl.append(s.tolist()); ll.append([0]*len(s))
    if not bl: return np.zeros((0,4),np.float32), np.zeros(0,np.float32)
    fb,fs,_ = weighted_boxes_fusion(bl,sl,ll,iou_thr=0.55,skip_box_thr=CAND_THRESH)
    fb = fb.copy(); fb[:,[0,2]]*=W; fb[:,[1,3]]*=H
    return fb.astype(np.float32), fs.astype(np.float32)


## 7. Activation-guided channel pruning (cls_subnet)

In [9]:
def _prune_subnet(model, seq, prune_frac, poison_xywh, tag):
    '''Prune channel phản hồi poison mạnh trong 1 subnet (cls hoặc bbox).'''
    convs=[m for m in seq if isinstance(m, nn.Conv2d)]
    stored={i:[] for i in range(len(convs))}; hooks=[]
    for i,ly in enumerate(convs):
        hooks.append(ly.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    dicts=[{'file_name':str(Path(UNLEARN_DIR)/f'{s}.png'),'image_id':int(s),'height':IMG_H,'width':IMG_W} for s in unlearn_imgs]
    with torch.no_grad():
        for d in dicts:
            im=load_image(d['file_name']); model([{'image':torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    total=0
    for li,layer in enumerate(convs):
        acts=stored[li]; C=acts[0].shape[1]; fg=torch.zeros(C); bg=torch.zeros(C); nf=nb_=0
        rng=np.random.default_rng(SEED)
        for act,d in zip(acts,dicts):
            a=act[0]; _,aH,aW=a.shape; sy,sx=aH/d['height'],aW/d['width']
            for (x,y,w,h) in poison_xywh.get(d['image_id'],[]):
                x1,y1=max(0,int(x*sx)),max(0,int(y*sy)); x2,y2=min(aW,int((x+w)*sx)+1),min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg+=a[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            for _ in range(max(1,len(poison_xywh.get(d['image_id'],[])))):
                ph,pw=max(1,int(16*sy)),max(1,int(16*sx)); ry,rx=rng.integers(0,max(1,aH-ph)),rng.integers(0,max(1,aW-pw))
                bg+=a[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        score=(fg-bg).numpy(); n_prune=max(1,int(len(score)*prune_frac)); bad=np.argsort(score)[-n_prune:]
        with torch.no_grad():
            layer.weight.data[bad]=0.0
            if layer.bias is not None: layer.bias.data[bad]=0.0
        total+=n_prune
    return total

def collect_and_prune(model, prune_frac, prune_box=True):
    model.eval(); total=0
    total += _prune_subnet(model, model.head.cls_subnet, prune_frac, poison_xywh, 'cls')
    if prune_box and hasattr(model.head,'bbox_subnet'):
        total += _prune_subnet(model, model.head.bbox_subnet, prune_frac, poison_xywh, 'box')
    return model, total

## 8. Baseline poisoned + neo LB

In [10]:
base_stats = eval_model_on_val(poisoned_predictor, VAL)
LB_ANCHOR_MACADD = 1.093   # local val_macadd của p05 (=LB 261.2)
print('POISONED baseline:', {k: round(v,4) for k,v in base_stats.items()})
print('Local neo (p05 = LB261.2): val_macadd =', LB_ANCHOR_MACADD)
print('-> Chỉ nộp NB4 nếu val_macadd < %.3f VÀ retention >= 0.6' % LB_ANCHOR_MACADD)

POISONED baseline: {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
Local neo (p05 = LB261.2): val_macadd = 1.093
-> Chỉ nộp NB4 nếu val_macadd < 1.093 VÀ retention >= 0.6


## 9. Retain set DÀY (in-distribution) — cải tiến #1
Sinh nhiều ảnh = nền trời thật + synthetic streak (BIẾT box = GT dương) + poison dán KHÔNG nhãn (forget). Nhãn dày (khác 0.25 box/ảnh gây collapse ở NB2). Model học: giữ detect streak, coi poison = background.

In [11]:
IMG_STORE = {}
def build_real_retain(n_test=250, retain_conf=0.4, n_poison_imgs=20, add_synth=True, seed=SEED):
    '''Retain set IN-DISTRIBUTION: nền ảnh test thật + pseudo-GT streak từ model gốc,
    TĂNG mật độ bằng synthetic streak có nhãn chuẩn (chống retain thưa gây under-train).
    Poison patch dán (không nhãn) = forget signal.'''
    rng = np.random.default_rng(seed); records = []; npos = 0; nposimg = 0
    pick = list(range(len(test_files))); rng.shuffle(pick); pick = pick[:n_test]
    for k in pick:
        p = test_files[k]; img = load_image(p); anns = []
        b, sc = predict_image(poisoned_predictor, img)
        rb = b[sc >= retain_conf] if len(b) else b
        for (x1,y1,x2,y2) in rb:
            x1=float(np.clip(x1,0,IMG_W)); y1=float(np.clip(y1,0,IMG_H))
            x2=float(np.clip(x2,0,IMG_W)); y2=float(np.clip(y2,0,IMG_H))
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[x1,y1,x2,y2],'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
        if add_synth:  # bơm thêm synthetic streak (nhãn CHẮC CHẮN đúng) lên nền test thật
            g = img[:,:,0].copy()
            for _ in range(rng.integers(1,4)):
                g, box = synth_streak_on(g, rng)
                x1,y1,x2,y2 = box
                if x2-x1>1 and y2-y1>1:
                    anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],
                                 'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
            img = np.repeat(g[:,:,None],3,axis=2)
        iid = 20_000_000 + int(k); IMG_STORE[iid] = img.astype(np.float32)
        records.append({'image_id': iid,'height':IMG_H,'width':IMG_W,'annotations':anns})
    # ảnh poison-forget từ unlearn set
    for j in range(n_poison_imgs):
        stem = stems[rng.integers(len(stems))]; img = unlearn_imgs[stem].copy(); anns=[]
        b, sc = predict_image(poisoned_predictor, img)
        pb = poison_boxes_by_stem.get(stem, np.zeros((0,4),np.float32))
        keep = sc >= retain_conf
        if len(b) and len(pb) and keep.any():
            keep = keep & (iou_matrix(b, pb).max(1) < POISON_OVERLAP_IOU)
        for (x1,y1,x2,y2) in (b[keep] if len(b) else b):
            if x2-x1>1 and y2-y1>1:
                anns.append({'bbox':[float(x1),float(y1),float(x2),float(y2)],'bbox_mode':BoxMode.XYXY_ABS,'category_id':0}); npos+=1
        for _ in range(rng.integers(1,3)):
            patch = poison_patches[rng.integers(len(poison_patches))]; ph,pw = patch.shape[:2]
            spot = rand_free_spot(stem, ph, pw, rng)
            if spot: paste_patch(img, patch, spot[0], spot[1])
        iid = 21_000_000 + j; IMG_STORE[iid] = img.astype(np.float32); nposimg += 1
        records.append({'image_id': iid,'height':IMG_H,'width':IMG_W,'annotations':anns})
    rng.shuffle(records)
    return records, npos, nposimg
IMG_STORE.clear()
_r,_np,_npi = build_real_retain(n_test=80, n_poison_imgs=10, seed=1)
print(f'probe retain-thật+synth: {len(_r)} ảnh | {_np} pseudo-GT streak ({_np/max(len(_r),1):.2f}/ảnh) | {_npi} ảnh poison-forget')
IMG_STORE.clear()

probe retain-thật+synth: 90 ảnh | 208 pseudo-GT streak (2.31/ảnh) | 10 ảnh poison-forget


## 10. FT-SAM optimizer + trainer (cải tiến #2 & #3)
SAM: leo gradient bán kính rho rồi cập nhật tại điểm nhiễu → co norm neuron backdoor (arXiv 2304.11823). EWC neo head về model gốc chống trôi. Backbone freeze.

In [12]:
RETAIN_DS = 'retain_dense'; _REC = {'r': None}
def register_dense(records):
    if RETAIN_DS in DatasetCatalog: DatasetCatalog.remove(RETAIN_DS)
    _REC['r'] = records; DatasetCatalog.register(RETAIN_DS, lambda: _REC['r'])
    MetadataCatalog.get(RETAIN_DS).set(thing_classes=['object'])
class DenseMapper(DatasetMapper):
    def __call__(self, dd):
        dd = copy.deepcopy(dd); im = IMG_STORE[dd['image_id']]
        dd['image'] = torch.as_tensor(im.transpose(2,0,1).copy())
        dd['instances'] = utils.annotations_to_instances(dd.get('annotations', []), im.shape[:2])
        return dd
def freeze_backbone(model):
    for n,p in model.named_parameters():
        if n.startswith('backbone.'): p.requires_grad = False
    return model
def make_ftsam_trainer(anchor_holder, ewc_lambda, rho):
    class T(DefaultTrainer):
        @classmethod
        def build_train_loader(cls, cfg):
            return build_detection_train_loader(cfg, mapper=DenseMapper(cfg,is_train=True,augmentations=[]),
                                                 dataset=DatasetCatalog.get(cfg.DATASETS.TRAIN[0]))
        @classmethod
        def build_model(cls, cfg):
            return freeze_backbone(super().build_model(cfg))
        def _losses(self, data):
            ld = self.model(data)
            if anchor_holder['w'] is not None:
                ewc = torch.tensor(0.0, device=DEVICE)
                for n,p in self.model.named_parameters():
                    if p.requires_grad and n in anchor_holder['w']:
                        ewc = ewc + ((p - anchor_holder['w'][n])**2).sum()
                ld['loss_ewc'] = ewc_lambda * ewc
            return sum(ld.values())
        def run_step(self):
            if not hasattr(self,'_it'): self._it = iter(self.data_loader)
            try: data = next(self._it)
            except StopIteration: self._it = iter(self.data_loader); data = next(self._it)
            params = [p for p in self.model.parameters() if p.requires_grad]
            # --- SAM ascent ---
            loss1 = self._losses(data); self.optimizer.zero_grad(); loss1.backward()
            gnorm = torch.norm(torch.stack([p.grad.norm() for p in params if p.grad is not None]))
            scale = rho / (gnorm + 1e-12); ew = []
            with torch.no_grad():
                for p in params:
                    if p.grad is None: ew.append(None); continue
                    e = p.grad * scale; p.add_(e); ew.append(e)
            # --- descent tại điểm nhiễu ---
            loss2 = self._losses(data); self.optimizer.zero_grad(); loss2.backward()
            with torch.no_grad():
                for p,e in zip(params, ew):
                    if e is not None: p.sub_(e)
            self.optimizer.step()
    return T

## 11. Train: prune nhẹ + FT-SAM trên retain-DÀY

In [13]:
def train_ftsam(prune_frac, n_retain, ewc_lambda, rho, lr, iters, seed, tag):
    out = WORK/tag; out.mkdir(parents=True, exist_ok=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    cfg0 = build_cfg(POISONED_WEIGHTS, output_dir=out)
    model = build_model(cfg0); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model.to(DEVICE)
    if prune_frac > 0: model, _ = collect_and_prune(model, prune_frac)
    pruned = str(out/'pruned.pth'); torch.save({'model': model.state_dict()}, pruned)
    anchor = {n:p.detach().clone() for n,p in model.named_parameters() if not n.startswith('backbone.')}
    holder = {'w': anchor}; del model; torch.cuda.empty_cache()
    IMG_STORE.clear()
    records, npos, _ = build_real_retain(n_test=n_retain, retain_conf=0.4, seed=seed); register_dense(records)
    cfg = build_cfg(pruned, output_dir=out)
    cfg.DATASETS.TRAIN=(RETAIN_DS,); cfg.DATASETS.TEST=(); cfg.DATALOADER.NUM_WORKERS=2
    cfg.DATALOADER.FILTER_EMPTY_ANNOTATIONS=False
    cfg.SOLVER.IMS_PER_BATCH=BATCH_SIZE; cfg.SOLVER.BASE_LR=lr; cfg.SOLVER.MAX_ITER=iters; cfg.SOLVER.STEPS=[]
    Tr = make_ftsam_trainer(holder, ewc_lambda, rho); tr = Tr(cfg); tr.resume_or_load(resume=False); tr.train()
    return str(out/'model_final.pth')

## 12. Sweep — chọn theo val_macadd (giữ streak + xóa poison)

In [14]:
SWEEP = [
    dict(prune_frac=0.05, n_retain=200, ewc_lambda=1000, rho=0.05, lr=1e-4, iters=150, seed=42, tag='s0'),
    dict(prune_frac=0.05, n_retain=300, ewc_lambda=2000, rho=0.05, lr=1e-4, iters=200, seed=42, tag='s1'),
    dict(prune_frac=0.05, n_retain=300, ewc_lambda=1000, rho=0.10, lr=1e-4, iters=200, seed=7,  tag='s2'),
    dict(prune_frac=0.10, n_retain=300, ewc_lambda=2000, rho=0.05, lr=1e-4, iters=200, seed=13, tag='s3'),
]
sweep_results = []
for spec in SWEEP:
    w = train_ftsam(**spec)
    pr = build_predictor(w, CAND_THRESH); stats = eval_model_on_val(pr, VAL); del pr; torch.cuda.empty_cache()
    sweep_results.append({**spec, 'weights': w, **stats})
    print(f"{spec['tag']}: firing={stats['poison_firing_rate']:.3f} "
          f"retention={stats['streak_retention']:.3f} val_macadd={stats['val_macadd']:.4f}")
sweep_df = pd.DataFrame(sweep_results).sort_values('val_macadd')
print('\n', sweep_df[['tag','prune_frac','n_retain','ewc_lambda','rho','iters','poison_firing_rate','streak_retention','val_macadd']].to_string(index=False))
print(f"\nPOISONED baseline: firing={base_stats['poison_firing_rate']:.3f} retention={base_stats['streak_retention']:.3f} val_macadd={base_stats['val_macadd']:.4f}")
print(f"neo p05 (LB261.2): val_macadd={LB_ANCHOR_MACADD}")

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/08 21:09:48 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

2026-07-08 21:10:18.112246: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783545018.277548      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783545018.323713      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783545018.699865      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783545018.699907      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783545018.699910      23 computation_placer.cc:177] computation placer alr

[07/08 21:10:57 d2.utils.events]:  eta: 0:02:27  iter: 39      time: 1.3435  last_time: 1.4387   lr: 2.6074e-05  max_mem: 1305M
[07/08 21:11:25 d2.utils.events]:  eta: 0:02:02  iter: 59      time: 1.3590  last_time: 1.3519   lr: 3.9394e-05  max_mem: 1305M
[07/08 21:11:51 d2.utils.events]:  eta: 0:01:33  iter: 79      time: 1.3535  last_time: 1.3410   lr: 5.2714e-05  max_mem: 1305M
[07/08 21:12:18 d2.utils.events]:  eta: 0:01:07  iter: 99      time: 1.3547  last_time: 1.3826   lr: 6.6034e-05  max_mem: 1305M
[07/08 21:12:46 d2.utils.events]:  eta: 0:00:40  iter: 119      time: 1.3576  last_time: 1.3600   lr: 7.9354e-05  max_mem: 1305M
[07/08 21:13:13 d2.utils.events]:  eta: 0:00:13  iter: 139      time: 1.3570  last_time: 1.3553   lr: 9.2674e-05  max_mem: 1305M
[07/08 21:13:27 d2.utils.events]:  eta: 0:00:00  iter: 149      time: 1.3564  last_time: 1.3454   lr: 9.9334e-05  max_mem: 1305M
[07/08 21:13:27 d2.engine.hooks]: Overall training speed: 148 iterations in 0:03:20 (1.3564 s / it)
[

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


...

s2: firing=0.270 retention=0.541 val_macadd=1.1987
[07/08 21:24:39 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/08 21:25:46 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/08 21:30:17 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/s3/model_final.pth ...
s3: firing=0.031 retention=0.270 val_macadd=1.2448

 tag  prune_frac  n_retain  ewc_lambda  rho  iters  poison_firing_rate  streak_retention  val_macadd
 s0        0.05       200        1000 0.05    150            0.288344          0.559748    1.185238
 s2        0.05       300        1000 0.10    200            0.269939          0.540881    1.198675
 s1        0.05       300        2000 0.05    200            0.263804          0.534591    1.209488
 s3        0.10       300        2000 0.05    200            0.030675          0.270440    1.244827

POISONED baseline: firing=0.969 retention=1.000 val_macadd=1.4549
neo p05 (LB261.2): val_macadd=1.093


## 13. Chọn cấu hình + model soup (weight averaging)

In [15]:
cands = [r for r in sweep_results if r['streak_retention'] >= 0.6 and r['val_macadd'] < LB_ANCHOR_MACADD]
pool = cands if cands else sweep_results
best = min(pool, key=lambda r: r['val_macadd'])
# model soup: trung bình weight các bản beat neo (Wortsman ICML'22)
soup_src = [r for r in sweep_results if r['val_macadd'] < LB_ANCHOR_MACADD] or [best]
if len(soup_src) >= 2:
    sds = [torch.load(r['weights'], map_location='cpu')['model'] for r in soup_src]
    avg = {k: sum(sd[k].float() for sd in sds)/len(sds) for k in sds[0]}
    soup_w = str(WORK/'soup.pth'); torch.save({'model': avg}, soup_w)
    pr = build_predictor(soup_w, CAND_THRESH); soup_stats = eval_model_on_val(pr, VAL); del pr; torch.cuda.empty_cache()
    print('SOUP:', {k: round(v,4) for k,v in soup_stats.items()})
    if soup_stats['streak_retention'] >= 0.6 and soup_stats['val_macadd'] < best['val_macadd']:
        best = {'tag':'soup','weights':soup_w, **soup_stats}
FINAL_WEIGHTS = best['weights']
FINAL_STATS = {k:best[k] for k in ('poison_firing_rate','streak_retention','val_macadd')}
print('CHỌN:', best['tag'], {k: round(float(v),4) for k,v in FINAL_STATS.items()})

CHỌN: s0 {'poison_firing_rate': 0.2883, 'streak_retention': 0.5597, 'val_macadd': 1.1852}


## 14. TTA D4+WBF (chỉ nhận nếu cải thiện validation)

In [16]:
final_pr = build_predictor(FINAL_WEIGHTS, CAND_THRESH)
tta_stats = eval_model_on_val(final_pr, VAL, tta=True)
print('no-TTA :', {k: round(float(v),4) for k,v in FINAL_STATS.items()})
print('TTA    :', {k: round(v,4) for k,v in tta_stats.items()})
USE_TTA = tta_stats['val_macadd'] <= FINAL_STATS['val_macadd']
print('-> dùng TTA:', USE_TTA)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/08 21:30:25 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/s0/model_final.pth ...
no-TTA : {'poison_firing_rate': 0.2883, 'streak_retention': 0.5597, 'val_macadd': 1.1852}
TTA    : {'poison_firing_rate': 0.2577, 'streak_retention': 0.566, 'val_macadd': 1.1791}
-> dùng TTA: True


## 15. Submission = output trực tiếp model đã de-poison

In [17]:
sample = pd.read_csv(SAMPLE_SUB, dtype={'image_id': str})
def fmt(b, s):
    parts = []
    for (x1,y1,x2,y2), sc in zip(b, s):
        x1,y1 = float(np.clip(x1,0,IMG_W)), float(np.clip(y1,0,IMG_H))
        x2,y2 = float(np.clip(x2,0,IMG_W)), float(np.clip(y2,0,IMG_H))
        w,h = x2-x1, y2-y1
        if w>0 and h>0 and sc>0: parts += [f'{float(sc):.6f}', f'{x1:.2f}', f'{y1:.2f}', f'{w:.2f}', f'{h:.2f}']
    return ' '.join(parts) if parts else ' '
pred = {}
for p in tqdm(test_files, desc='infer test'):
    img = load_image(p)
    b,s = predict_tta(final_pr, img) if USE_TTA else predict_image(final_pr, img)
    keep = s > SUBMIT_THRESH; pred[p.stem] = (b[keep], s[keep])
sub = sample.copy()
sub['prediction_string'] = sub['image_id'].map(lambda i: fmt(*pred.get(str(i), E)))
sub['prediction_string'] = sub['prediction_string'].fillna(' ')
sub.loc[sub['prediction_string'].str.strip()=='', 'prediction_string'] = ' '
sub.to_csv(SUBMISSION_PATH, index=False)
total = sum(len(s) for _,s in pred.values()); npred = int((sub['prediction_string'].str.strip()!='').sum())
print(f'saved {SUBMISSION_PATH} | rows={len(sub)} | dets={total} ({total/len(test_files):.3f}/img) | imgs w/ preds: {npred}')
print('so sánh: poisoned ~4.1/img | clean mục tiêu ~1.5/img | p05(LB261)=0.371/img')

infer test: 100%|██████████| 2000/2000 [27:36<00:00,  1.21it/s]

saved /kaggle/working/submission.csv | rows=2000 | dets=687 (0.344/img) | imgs w/ preds: 583
so sánh: poisoned ~4.1/img | clean mục tiêu ~1.5/img | p05(LB261)=0.371/img


## 16. Tổng kết + quy tắc nộp

In [18]:
print('='*60)
print('POISONED   :', {k: round(v,4) for k,v in base_stats.items()})
print('DE-POISONED:', {k: round(float(v),4) for k,v in FINAL_STATS.items()})
print('neo p05    : val_macadd=%.3f (LB=261.2)' % LB_ANCHOR_MACADD)
better = FINAL_STATS['val_macadd'] < LB_ANCHOR_MACADD
ret_ok = FINAL_STATS['streak_retention'] >= 0.6
dets_per_img = total/len(test_files)
ok = better and ret_ok and (0.5 <= dets_per_img <= 3.5)
print('\nĐẠT TIÊU CHÍ NỘP:', ok)
if ok:
    print(' -> NÊN nộp: đánh bại neo LB261 trên local, retention giữ, dets/img hợp lý.')
else:
    print(' -> CHƯA nộp (không tốt hơn neo p05). Giữ p05 là bản tốt nhất.')
    print('    Tune: tăng n_retain/iters nếu retention thấp; tăng rho/ewc nếu firing còn cao;')
    print('          xét module-localized reset (box_subnet) theo arXiv 2409.16057.')

POISONED   : {'poison_firing_rate': 0.9693, 'streak_retention': 1.0, 'val_macadd': 1.4549}
DE-POISONED: {'poison_firing_rate': 0.2883, 'streak_retention': 0.5597, 'val_macadd': 1.1852}
neo p05    : val_macadd=1.093 (LB=261.2)

ĐẠT TIÊU CHÍ NỘP: False
 -> CHƯA nộp (không tốt hơn neo p05). Giữ p05 là bản tốt nhất.
    Tune: tăng n_retain/iters nếu retention thấp; tăng rho/ewc nếu firing còn cao;
          xét module-localized reset (box_subnet) theo arXiv 2409.16057.
